# Lagrangian Tracker Validation: Synthetic Alfvén Wave

End-to-end validation of the Lagrangian trajectory tracer using synthetic simulation data
with a known shear Alfvén wave.

**Setup:**
- Uniform background: **B** = B0 ŷ (magnetic field along +y), **E** = −E0 ẑ (electric field along −z)
- E×B drift: v_x = E0/B0 in +x, v_y = 0
- Superimposed shear Alfvén wave propagating along +y (along B0)

**Key expectation:** The E×B drift is along x, but the wave propagates along y.
Since the drift has no component along the wave propagation direction,
the Lagrangian (co-moving) frequency should equal the Eulerian (lab-frame) frequency.
Both PSDs of Bx should show a peak at f = ω/(2π).

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from reconn_wave_power.spectrum import compute_psd_time
from reconn_wave_power.lagrangian import (
    compute_exb_velocity,
    trace_trajectory,
    sample_along_trajectory,
    lagrangian_psd,
)

%matplotlib inline

## Wave & Grid Parameters

In [ ]:
# Background fields
B0 = 1.0          # Background magnetic field magnitude (along +y)
E0 = 0.1          # Background electric field magnitude (along -z)
V_A = 1.0         # Alfven speed

# Wave parameters
LX = 32.0         # Domain size in x
LY = 32.0         # Domain size in y
WAVELENGTH = LY / 2  # 2 wavelengths fit in the y-domain
K_Y = 2 * np.pi / WAVELENGTH
OMEGA = K_Y * V_A  # Dispersion relation: omega = k_y * v_A
F0 = OMEGA / (2 * np.pi)  # Expected frequency in cycles/unit time
DB = B0 / 10       # Wave amplitude (small perturbation)

# Grid parameters
NX = 64
NY = 64
DX = LX / NX       # = 0.5
DY = LY / NY       # = 0.5

# Time parameters
DT_SIM = 0.05      # Internal simulation timestep
DT_OUTPUT = 0.1    # Output cadence (every 2nd internal step)
NT = 480           # Number of output frames (3 full wave periods)

# PSD segment length chosen so f0 lands exactly on a frequency bin:
# df = 1 / (NPERSEG * DT_OUTPUT) = 1/16 = 0.0625 = f0
NPERSEG = 160

print(f"Domain: {LX} x {LY}, grid: {NX} x {NY}, dx = {DX}, dy = {DY}")
print(f"Output cadence: {DT_OUTPUT} (sim dt = {DT_SIM}, differs from output cadence)")
print(f"Time range: 0 to {NT * DT_OUTPUT:.1f} ({NT} frames)")
print(f"")
print(f"B0 = {B0}, E0 = {E0}, drift speed = E0/B0 = {E0/B0}")
print(f"Wavelength = {WAVELENGTH}, k_y = {K_Y:.4f} rad/unit")
print(f"omega = {OMEGA:.4f} rad/unit time, f0 = {F0:.4f} cycles/unit time")
print(f"Wave period T = {WAVELENGTH/V_A:.1f}, {NT*DT_OUTPUT/(WAVELENGTH/V_A):.0f} full periods in dataset")
print(f"dB/B0 = {DB/B0}")
print(f"PSD nperseg = {NPERSEG}, df = {1/(NPERSEG*DT_OUTPUT):.4f}")

## Build Synthetic Dataset

In [ ]:
x = np.arange(NX) * DX
y = np.arange(NY) * DY
t = np.arange(NT) * DT_OUTPUT

# Broadcast y and t for wave computation: shape (nt, 1, ny)
Y = y[np.newaxis, np.newaxis, :]   # (1, 1, ny)
T = t[:, np.newaxis, np.newaxis]   # (nt, 1, 1)
phase = K_Y * Y - OMEGA * T        # (nt, 1, ny), broadcasts over x

# Build all 6 field components with shape (nt, nx, ny)
wave = DB * np.sin(phase)           # (nt, 1, ny) -> broadcasts to (nt, nx, ny)
zeros = np.zeros((NT, NX, NY))

ds = xr.Dataset(
    {
        "Bx": (("time", "x", "y"), np.broadcast_to(wave, (NT, NX, NY)).copy()),
        "By": (("time", "x", "y"), np.full((NT, NX, NY), B0)),
        "Bz": (("time", "x", "y"), zeros.copy()),
        "Ex": (("time", "x", "y"), zeros.copy()),
        "Ey": (("time", "x", "y"), zeros.copy()),
        "Ez": (("time", "x", "y"), np.broadcast_to(-E0 + V_A * wave, (NT, NX, NY)).copy()),
    },
    coords={"time": t, "x": x, "y": y},
)

print(ds)

## 1. Field Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

t_snap = 0

# Bx snapshot
ax = axes[0]
bx_snap = ds["Bx"].isel(time=t_snap).values.T
im = ax.pcolormesh(x, y, bx_snap, cmap="RdBu_r", shading="auto")
plt.colorbar(im, ax=ax, label="Bx")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title(f"Bx(x, y) at t = {t[t_snap]:.1f}")
ax.set_aspect("equal")

# Ez snapshot
ax = axes[1]
ez_snap = ds["Ez"].isel(time=t_snap).values.T
im = ax.pcolormesh(x, y, ez_snap, cmap="RdBu_r", shading="auto")
plt.colorbar(im, ax=ax, label="Ez")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title(f"Ez(x, y) at t = {t[t_snap]:.1f}")
ax.set_aspect("equal")

plt.tight_layout()
plt.show()

## 2. E×B Drift Velocity

In [ ]:
vx, vy = compute_exb_velocity(ds, time_idx=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
im = ax.pcolormesh(x, y, vx.values.T, cmap="RdBu_r", shading="auto")
plt.colorbar(im, ax=ax, label="vx")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("E×B drift: vx")
ax.set_aspect("equal")

ax = axes[1]
im = ax.pcolormesh(x, y, vy.values.T, cmap="RdBu_r", shading="auto")
plt.colorbar(im, ax=ax, label="vy")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("E×B drift: vy")
ax.set_aspect("equal")

plt.tight_layout()
plt.show()

print(f"Mean vx = {float(vx.mean()):.6f}  (expected E0/B0 = {E0/B0:.4f})")
print(f"Mean vy = {float(vy.mean()):.6f}  (expected ~ 0)")

## 3. Trajectory Tracing

In [ ]:
# Trace trajectories from several starting positions
starts = [
    (4.0, 4.0),
    (4.0, 12.0),
    (16.0, 8.0),
    (28.0, 24.0),
]

trajectories = []
for x0, y0 in starts:
    xt, yt, tt = trace_trajectory(ds, x0, y0, t0_idx=0)
    trajectories.append((xt, yt, tt))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# x(t)
ax = axes[0]
for i, (xt, yt, tt) in enumerate(trajectories):
    ax.plot(tt, xt, label=f"start ({starts[i][0]}, {starts[i][1]})")
ax.set_xlabel("Time")
ax.set_ylabel("x position")
ax.set_title("x(t) — should show linear drift with small wiggles")
ax.legend(fontsize=8)

# y(t)
ax = axes[1]
for i, (xt, yt, tt) in enumerate(trajectories):
    ax.plot(tt, yt, label=f"start ({starts[i][0]}, {starts[i][1]})")
ax.set_xlabel("Time")
ax.set_ylabel("y position")
ax.set_title("y(t) — small wave-induced oscillations")
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 4. Field Sampling Along Trajectory

In [ ]:
# Sample Bx along the first trajectory
xt, yt, tt = trajectories[0]
bx_sampled = sample_along_trajectory(ds, "Bx", xt, yt, tt)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(bx_sampled.coords["time"].values, bx_sampled.values, lw=0.8)
ax.set_xlabel("Time")
ax.set_ylabel("Bx")
ax.set_title("Bx sampled along Lagrangian trajectory")
ax.axhline(0, color="gray", lw=0.5, ls="--")
plt.tight_layout()
plt.show()

print(f"Expected wave period: {WAVELENGTH/V_A:.1f}")
print(f"Expected amplitude: +/- {DB:.2f}")

## 5. Lagrangian vs Eulerian PSD

In [ ]:
# --- Eulerian PSD: Bx time series at a fixed point ---
ix_fixed = NX // 4
iy_fixed = NY // 4
bx_euler = ds["Bx"].isel(x=ix_fixed, y=iy_fixed)
f_euler, pxx_euler = compute_psd_time(bx_euler, dt=DT_OUTPUT, method="welch", nperseg=NPERSEG)

# --- Lagrangian PSD ---
x0_lag, y0_lag = 4.0, 4.0
f_lag, pxx_lag, traj_info = lagrangian_psd(
    ds, "Bx", x0_lag, y0_lag, t0_idx=0, dt=DT_OUTPUT, method="fft", nperseg=NPERSEG,
)

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Eulerian
ax = axes[0]
ax.semilogy(f_euler, pxx_euler, "b-", lw=1)
ax.axvline(F0, color="r", ls="--", lw=2, label=f"Expected f0 = {F0:.4f}")
ax.set_xlabel("Frequency [cycles/unit time]")
ax.set_ylabel("PSD")
ax.set_title("Eulerian PSD (fixed point)")
ax.legend()
ax.set_xlim(0, 0.5)

# Lagrangian
ax = axes[1]
ax.semilogy(f_lag, pxx_lag, "g-", lw=1)
ax.axvline(F0, color="r", ls="--", lw=2, label=f"Expected f0 = {F0:.4f}")
ax.set_xlabel("Frequency [cycles/unit time]")
ax.set_ylabel("PSD")
ax.set_title("Lagrangian PSD (co-moving frame)")
ax.legend()
ax.set_xlim(0, 0.5)

plt.tight_layout()
plt.show()

# Verify peaks
euler_peak_f = f_euler[np.argmax(pxx_euler)]
lag_peak_f = f_lag[np.argmax(pxx_lag)]
print(f"Expected frequency: f0 = {F0:.4f} cycles/unit time")
print(f"Eulerian peak:  f = {euler_peak_f:.4f}")
print(f"Lagrangian peak: f = {lag_peak_f:.4f}")
print(f"Match: Eulerian and Lagrangian frequencies agree "
      f"(drift perpendicular to wave propagation)")

## Summary

This notebook validates the Lagrangian trajectory tracer against a known analytic solution:

1. **E×B drift is correct** — mean vx matches E0/B0, vy is near zero
2. **Trajectories follow expected drift** — linear x-drift with small wave-induced perturbations
3. **Bx along trajectory shows wave oscillation** — clear sinusoidal signal at the Alfvén wave frequency
4. **Lagrangian PSD recovers the correct frequency** — peak at f0 = ω/(2π)
5. **Lagrangian and Eulerian frequencies match** — as expected, since the E×B drift is perpendicular to the wave propagation direction

---

# Part 2: Random Correlated Fluctuations

In contrast to the clean Alfvén wave above, here we construct synthetic E and B fields
from **spatially correlated random noise** with a correlation length of 5 simulation length units.
The fields are "frozen" (static in time) so trajectories trace through a fixed turbulent landscape.

**Setup:**
- Background: Bz = B0 (out-of-plane, ensures |B| > 0)
- All 6 field components receive independent Gaussian random fluctuations
- Spatial correlations imposed via a Gaussian spectral filter: P(k) ∝ exp(−k² L_c² / 4)
- Correlation length L_c = 5.0 grid units
- Fluctuation amplitude: δB, δE ~ 0.1 B0
- E×B drift: vx = Ey·Bz/|B|², vy = −Ex·Bz/|B|² (dominated by out-of-plane B)

In [ ]:
# --- Parameters for random correlated fields ---
L_C = 5.0          # Correlation length (simulation length units)
DB_RMS = 0.1       # RMS amplitude of magnetic fluctuations
DE_RMS = 0.1       # RMS amplitude of electric fluctuations
NT_RAND = 480      # Same number of output frames
SEED = 42

def correlated_random_field(nx, ny, dx, dy, L_c, rng):
    """Generate a 2D spatially correlated Gaussian random field.

    Uses a Gaussian spectral filter: P(k) ~ exp(-k^2 L_c^2 / 4)
    so the real-space correlation function is ~ exp(-r^2 / L_c^2).
    """
    # Wavenumber grids
    kx = np.fft.fftfreq(nx, d=dx) * 2 * np.pi
    ky = np.fft.fftfreq(ny, d=dy) * 2 * np.pi
    KX, KY = np.meshgrid(kx, ky, indexing="ij")
    k2 = KX**2 + KY**2

    # Gaussian spectral envelope (amplitude filter)
    envelope = np.exp(-k2 * L_c**2 / 4)

    # White noise in Fourier space, filtered
    noise_k = np.fft.fft2(rng.standard_normal((nx, ny)))
    field = np.real(np.fft.ifft2(noise_k * envelope))

    # Normalize to unit variance then scale
    field /= field.std()
    return field

rng = np.random.default_rng(SEED)

# Generate one random 2D pattern per field component, frozen in time
field_names = ["Bx", "By_fluct", "Bz", "Ex", "Ey", "Ez"]
random_patterns = {}
for name in field_names:
    random_patterns[name] = correlated_random_field(NX, NY, DX, DY, L_C, rng)

print(f"Correlation length: {L_C}")
print(f"Fluctuation amplitudes: dB_rms = {DB_RMS}, dE_rms = {DE_RMS}")
print(f"Pattern shape: {random_patterns['Bx'].shape}")

### Build Random-Field Dataset

Each 2D pattern is tiled across all timesteps (frozen turbulence).
Bz = B0 + fluctuation ensures |B| stays positive (out-of-plane background field).

In [ ]:
t_rand = np.arange(NT_RAND) * DT_OUTPUT

# Tile each 2D pattern to (nt, nx, ny) — frozen in time
def tile_to_3d(pattern_2d, nt):
    return np.broadcast_to(pattern_2d[np.newaxis, :, :], (nt, NX, NY)).copy()

ds_rand = xr.Dataset(
    {
        "Bx": (("time", "x", "y"), tile_to_3d(DB_RMS * random_patterns["Bx"], NT_RAND)),
        "By": (("time", "x", "y"), tile_to_3d(DB_RMS * random_patterns["By_fluct"], NT_RAND)),
        "Bz": (("time", "x", "y"), tile_to_3d(B0 + DB_RMS * random_patterns["Bz"], NT_RAND)),
        "Ex": (("time", "x", "y"), tile_to_3d(DE_RMS * random_patterns["Ex"], NT_RAND)),
        "Ey": (("time", "x", "y"), tile_to_3d(DE_RMS * random_patterns["Ey"], NT_RAND)),
        "Ez": (("time", "x", "y"), tile_to_3d(DE_RMS * random_patterns["Ez"], NT_RAND)),
    },
    coords={"time": t_rand, "x": x, "y": y},
)

print(ds_rand)

### Random Field Visualization

Snapshot of all 6 field components showing spatially correlated structure at L_c = 5.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for ax, comp in zip(axes.flat, ["Bx", "By", "Bz", "Ex", "Ey", "Ez"]):
    snap = ds_rand[comp].isel(time=0).values.T
    im = ax.pcolormesh(x, y, snap, cmap="RdBu_r", shading="auto")
    plt.colorbar(im, ax=ax)
    ax.set_title(comp)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_aspect("equal")

plt.suptitle(f"Random correlated fields (L_c = {L_C})", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### E×B Drift from Random Fields

In [ ]:
vx_rand, vy_rand = compute_exb_velocity(ds_rand, time_idx=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
im = ax.pcolormesh(x, y, vx_rand.values.T, cmap="RdBu_r", shading="auto")
plt.colorbar(im, ax=ax, label="vx")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("E×B drift: vx (random fields)")
ax.set_aspect("equal")

ax = axes[1]
im = ax.pcolormesh(x, y, vy_rand.values.T, cmap="RdBu_r", shading="auto")
plt.colorbar(im, ax=ax, label="vy")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("E×B drift: vy (random fields)")
ax.set_aspect("equal")

plt.tight_layout()
plt.show()

print(f"vx range: [{float(vx_rand.min()):.4f}, {float(vx_rand.max()):.4f}]")
print(f"vy range: [{float(vy_rand.min()):.4f}, {float(vy_rand.max()):.4f}]")

### Trajectories in Random Fields

With frozen random fluctuations the E×B velocity is spatially structured but constant in time.
Trajectories should wander through the correlated velocity landscape, wrapping periodically.

In [ ]:
# Trace trajectories from a grid of starting positions
starts_rand = [
    (4.0, 4.0),
    (4.0, 16.0),
    (4.0, 28.0),
    (16.0, 4.0),
    (16.0, 16.0),
    (16.0, 28.0),
    (28.0, 4.0),
    (28.0, 16.0),
]

trajs_rand = []
for x0, y0 in starts_rand:
    xt, yt, tt = trace_trajectory(ds_rand, x0, y0, t0_idx=0)
    trajs_rand.append((xt, yt, tt))

# --- x(t) and y(t) ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for i, (xt, yt, tt) in enumerate(trajs_rand):
    ax.plot(tt, xt, lw=0.8, label=f"({starts_rand[i][0]:.0f}, {starts_rand[i][1]:.0f})")
ax.set_xlabel("Time")
ax.set_ylabel("x position")
ax.set_title("x(t) in random fields")
ax.legend(fontsize=7, ncol=2)

ax = axes[1]
for i, (xt, yt, tt) in enumerate(trajs_rand):
    ax.plot(tt, yt, lw=0.8, label=f"({starts_rand[i][0]:.0f}, {starts_rand[i][1]:.0f})")
ax.set_xlabel("Time")
ax.set_ylabel("y position")
ax.set_title("y(t) in random fields")
ax.legend(fontsize=7, ncol=2)

plt.tight_layout()
plt.show()

In [ ]:
# --- Trajectories overlaid on E×B speed ---
speed = np.sqrt(vx_rand.values**2 + vy_rand.values**2)

fig, ax = plt.subplots(figsize=(8, 8))
im = ax.pcolormesh(x, y, speed.T, cmap="viridis", shading="auto")
plt.colorbar(im, ax=ax, label="|v_ExB|")

for i, (xt, yt, tt) in enumerate(trajs_rand):
    ax.plot(xt, yt, lw=1.0, alpha=0.8)
    ax.plot(xt[0], yt[0], "o", ms=6, color="white", zorder=5)

ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Trajectories overlaid on |v_ExB| (random correlated fields)")
ax.set_aspect("equal")
ax.set_xlim(0, LX)
ax.set_ylim(0, LY)
plt.tight_layout()
plt.show()

In [ ]:
# --- Trajectories overlaid on vx and vy components ---
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for ax, vel, label in zip(axes, [vx_rand, vy_rand], ["vx", "vy"]):
    im = ax.pcolormesh(x, y, vel.values.T, cmap="RdBu_r", shading="auto", vmin=-.5, vmax=.5)
    plt.colorbar(im, ax=ax, label=label)

    for i, (xt, yt, tt) in enumerate(trajs_rand):
        ax.plot(xt, yt, "k-", lw=0.8, alpha=0.7)
        ax.plot(xt[0], yt[0], "o", ms=5, color="white", mec="black", mew=0.5, zorder=5)

    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_title(f"Trajectories on E×B {label}")
    ax.set_aspect("equal")
    ax.set_xlim(0, LX)
    ax.set_ylim(0, LY)

plt.tight_layout()
plt.show()